# BeamSearchCodeAgent on a single GAIA question

Beam search over naiveReAct-style full (thought, code) resamples: every live beam samples
`branch_factor` independent continuations each round, all candidates are pruned down to
`beam_width` survivors by `Beam.pruning_score` (log P(judge=yes) + thought/code generation
logprob, by the chain rule, averaged per step -- see TODO.md), until every beam is `is_final`
or `max_steps` is hit.

Set the config below and run top to bottom. Cell 7 prints the full round-by-round beam tree
(thought/code/observation/scores per candidate, which ones survived pruning) and can be re-run
on its own to re-view a completed run without calling the agent again.


In [1]:
# Pick up edits to the editable-installed smolagents without a kernel restart.
%load_ext autoreload
%autoreload 2

# ===================== experiment configuration =====================
MODEL = "gpt-4o"                      # key of MODELS below
QUESTION_INDEX = 0                 # single GAIA validation index to run
MAX_STEPS = 20
BEAM_WIDTH = 2
BRANCH_FACTOR = 2
RUN_TAG = "_inspect"                  # used only for the saved-trajectory filename in the last cell
# ====================================================================

MODELS = {
    "gpt-4o": "openai",
    "gpt-5.4-mini": "openai",
    "Qwen/Qwen3.5-9B": "together",
    "Qwen/Qwen3.7-Plus": "together",
}
ENABLE_THINKING = False  # Together-served open-weight models only

assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; pick one of {list(MODELS)}"


In [2]:
import os
import sys

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = MODEL
provider = MODELS[MODEL]

if provider == "openai":
    model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
else:
    # Together needs enable_thinking nested in chat_template_kwargs for vLLM-served models (see naiveReAct).
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        client_kwargs={"timeout": 300.0},  # bound a stalled call instead of hanging forever
    )

print(f"{model_name} via {provider}")


gpt-4o via openai


In [3]:
# Standard GAIA tool stack, same as naiveReAct.ipynb / traceletReAct_single.ipynb.
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)


web_search: LeakFilteredDuckDuckGoSearchTool at 1.0 q/s


/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [4]:
from smolagents.beam_search_agent import BeamSearchCodeAgent
from smolagents.monitoring import LogLevel

# BeamSearchCodeAgent always uses the default code_agent.yaml (plain ReAct) -- unlike
# TraceletCodeAgent it has no skeleton/fill-in protocol prompt.
agent = BeamSearchCodeAgent(
    tools=tools,
    model=model,
    max_steps=MAX_STEPS,
    beam_width=BEAM_WIDTH,
    branch_factor=BRANCH_FACTOR,
    verbosity_level=LogLevel.ERROR,  # per-round detail comes from the instrumentation below instead
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
)


In [5]:
# BeamSearchCodeAgent has no built-in progress logging (by design, until this notebook decides
# what's worth keeping permanently) -- instrument _expand_beam/_prune to record a round-by-round
# beam tree that cell 7 renders, and that survives for re-inspection without re-running the agent.
rounds = []
_current_round_expansions = []

_orig_expand_beam = BeamSearchCodeAgent._expand_beam
_orig_prune = BeamSearchCodeAgent._prune


def _record_expand(self, beam):
    children = _orig_expand_beam(self, beam)
    child_records = []
    for c in children:
        step = c.memory.steps[-1]
        thought = (step.model_output or "").split(self.code_block_tags[0], 1)[0].strip()
        child_records.append({
            "beam_id": c.beam_id,
            "errored": c.errored_this_step,
            "is_final": c.is_final,
            "p_yes": c.expectation_scores[-1] if c.expectation_scores else None,
            "pruning_score": c.pruning_score,
            "thought": thought,
            "code": step.code_action or "",
            "observation": step.observations or "",
        })
    _current_round_expansions.append({
        "parent_id": beam.beam_id,
        "parent_pruning_score": beam.pruning_score,
        "children": child_records,
    })
    return children


def _record_prune(self, beams):
    survivors = _orig_prune(self, beams)
    rounds.append({
        "n": len(rounds) + 1,
        "expansions": list(_current_round_expansions),
        "kept": [b.beam_id for b in survivors],
    })
    _current_round_expansions.clear()
    return survivors


BeamSearchCodeAgent._expand_beam = _record_expand
BeamSearchCodeAgent._prune = _record_prune


In [6]:
# --- Alternative to the cell above: load an already-run trajectory instead of calling the
# agent again -- no API cost.
# Two source formats are supported:
#   LOAD_FROM_FILE  -- a single-file dry-run trajectory, e.g. "beam_dryrun_q6.json"
#   LOAD_FROM_SWEEP -- (run_name, index) from a full sweep, e.g.
#                      ("beamsearch_bw2_bf2_react_gpt-4o", 0) for baseline/beamsearch_bw2_bf2_react_gpt-4o/0.pkl
#                      + baseline/beamsearch_bw2_bf2_react_gpt-4o_trees/0.json
# Set ONE of the two below and re-run this cell, then cell 7 below; skip the "run one GAIA
# question" cell above if you're just revisiting a saved trajectory.
LOAD_FROM_FILE = None  # e.g. "beam_dryrun_q9.json"
LOAD_FROM_SWEEP = ("beamsearch_bw2_bf2_react_gpt-4o", 0)  # e.g. (run_name, index) -- or None

if LOAD_FROM_FILE:
    from common_setup import question_scorer
    import json as _json

    with open(LOAD_FROM_FILE) as f:
        _saved = _json.load(f)
    example = _saved["example"]
    prediction = _saved["prediction"]
    rounds = _saved["rounds"]

    print(f"loaded {LOAD_FROM_FILE}")
elif LOAD_FROM_SWEEP:
    from common_setup import question_scorer
    import json as _json
    import pickle as _pickle

    _run_name, _index = LOAD_FROM_SWEEP
    with open(f"{_run_name}/{_index}.pkl", "rb") as f:
        example = _pickle.load(f)  # evaluate_agent's result dict already has question/true_answer/task_id
    with open(f"{_run_name}_trees/{_index}.json") as f:
        rounds = _json.load(f)
    prediction = example["prediction"]

    print(f"loaded {_run_name}/{_index} (pkl + tree)")

if LOAD_FROM_FILE or LOAD_FROM_SWEEP:
    print(f"task_id  : {example.get('task_id')}")
    print(f"question : {example['question']}")
    print(f"true     : {example['true_answer']}")
    print(f"prediction : {prediction!r}")
    print(f"correct    : {question_scorer(str(prediction), example['true_answer'])}")


loaded beamsearch_bw2_bf2_react_gpt-4o/0 (pkl + tree)
task_id  : c61d22de-5f6c-4958-a7f6-5e9707bd3466
question : A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?
true     : egalitarian
prediction : 'egalitarian or hierarchical'
correct    : False


In [7]:
if LOAD_FROM_FILE or LOAD_FROM_SWEEP:
    print(f"LOAD_FROM_FILE is set ({LOAD_FROM_FILE!r}) -- skipping the live agent run.")
else:
    from common_setup import load_gaia_dataset, question_scorer

    eval_ds = load_gaia_dataset(set_to_run="validation")
    example = eval_ds.to_list()[QUESTION_INDEX]
    assert not example["file_name"], f"q{QUESTION_INDEX} has an attachment; attachment preprocessing is skipped here"

    print(f"Q{QUESTION_INDEX}  task_id={example.get('task_id')}")
    print(f"question : {example['question']}")
    print(f"true     : {example['true_answer']}")
    print("=" * 100)

    prediction = agent.run(example["question"])

    print("=" * 100)
    print(f"prediction : {prediction!r}")
    print(f"true       : {example['true_answer']!r}")
    print(f"correct    : {question_scorer(str(prediction), example['true_answer'])}")


LOAD_FROM_FILE is set (None) -- skipping the live agent run.


In [8]:
# Beam tree, rendered as an actual tree (ancestry via beam_id, not a flat round-by-round list).
# Re-run this cell alone to re-view a completed run without re-running the agent.
TRUNC = 200


def _trunc(s, n=TRUNC):
    s = (s or "").strip()
    return s if len(s) <= n else s[: n - 1] + "\u2026"


# --- Rebuild the tree from `rounds` (built by the instrumentation in the previous cell) ---
nodes = {
    "0": {
        "beam_id": "0",
        "errored": False,
        "is_final": False,
        "p_yes": None,
        "pruning_score": 0.0,
        "thought": "(root -- task given, no step taken yet)",
        "code": "",
        "observation": "",
    }
}
children_of = {}
for r in rounds:
    for exp in r["expansions"]:
        children_of.setdefault(exp["parent_id"], [])
        for c in exp["children"]:
            nodes[c["beam_id"]] = c
            children_of[exp["parent_id"]].append(c["beam_id"])

# --- Replay run()'s own winner-selection rule against the final surviving pool, so the tree can
# mark exactly which leaf the agent's answer came from ---
last_round_kept = set(rounds[-1]["kept"]) if rounds else set()
final_frontier = [nodes[bid] for bid in last_round_kept]
finished = [n for n in final_frontier if n["is_final"]]
winner_id = max(finished if finished else final_frontier, key=lambda n: n["pruning_score"])["beam_id"] if final_frontier else None


def _status(beam_id):
    node = nodes[beam_id]
    tags = []
    if beam_id == winner_id:
        tags.append("\u2605 WINNER")
    if node["errored"]:
        tags.append("ERRORED")
    if node["is_final"]:
        tags.append("FINAL")
    elif beam_id not in children_of:
        tags.append("UNFINISHED (search ended)" if beam_id in last_round_kept else "CUT (pruned)")
    return f" [{', '.join(tags)}]" if tags else ""


def _fmt(value):
    return f"{value:.3f}" if value is not None else "n/a"


def _header(beam_id):
    node = nodes[beam_id]
    p_yes = _fmt(node["p_yes"])
    thought_lp = _fmt(node.get("thought_logprob"))
    code_lp = _fmt(node.get("code_logprob"))
    return (
        f"{beam_id}{_status(beam_id)}  p_yes={p_yes}  thought_lp={thought_lp}  code_lp={code_lp}  "
        f"pruning_score={node['pruning_score']:.3f}"
    )


def _content_lines(beam_id):
    node = nodes[beam_id]
    lines = []
    if node["thought"]:
        lines.append(f"thought: {_trunc(node['thought'])}")
    if node["code"]:
        lines.append(f"code   : {_trunc(node['code'])}")
    if node["observation"]:
        lines.append(f"obs    : {_trunc(node['observation'])}")
    return lines


def print_tree(beam_id, prefix="", connector=""):
    print(prefix + connector + _header(beam_id))
    child_prefix = prefix + ("    " if connector in ("", "\u2514\u2500\u2500 ") else "\u2502   ")
    for line in _content_lines(beam_id):
        print(child_prefix + line)
    kids = children_of.get(beam_id, [])
    for i, kid in enumerate(kids):
        is_last = i == len(kids) - 1
        print_tree(kid, child_prefix, "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 ")


print_tree("0")

print("\n" + "=" * 100)
print(f"prediction : {prediction!r}")
print(f"true       : {example['true_answer']!r}")
print(f"correct    : {question_scorer(str(prediction), example['true_answer'])}")


0  p_yes=n/a  thought_lp=n/a  code_lp=n/a  pruning_score=0.000
    thought: (root -- task given, no step taken yet)
    ├── 0.0  p_yes=0.007  thought_lp=-0.424  code_lp=-0.334  pruning_score=-5.776
    │   thought: **Thought:** First, I will perform a web search to locate the paper about AI regulation submitted to arXiv.org in June 2022 in order to identify the figure with three axes. I will then examine this f…
    │   code   : query = "AI regulation site:arxiv.org submitted June 2022"
ai_regulation_pages = web_search(query=query)
print(ai_regulation_pages)
    │   obs    : Execution logs:
## Search Results

[arxiv.org/pdf/2210.02747](https://arxiv.org/pdf/2210.02747)
Lipman et al., “Flow Matching for Generative Modeling” (2022).

[[2212.08073] Constitutional AI: Harmle…
    │   ├── 0.0.0  p_yes=0.024  thought_lp=-0.601  code_lp=-0.163  pruning_score=-5.141
    │   │   thought: **Thought:** From the search results, it appears the potential paper about AI regulation that was submitted 

In [9]:
# Compile the currently-loaded trace into the flip/if-then-else/factor/resample_top_k
# probabilistic program (see compile_trace_to_program.py) -- works whether `rounds` came from
# the live run above or one of the LOAD_FROM_FILE / LOAD_FROM_SWEEP options.
from compile_trace_to_program import compile_trace

program = compile_trace(
    rounds,
    question=example.get("question"),
    true_answer=example.get("true_answer"),
    prediction=prediction,
)
print(program)


# question: A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?
# true_answer: egalitarian

# ============ ROUND 1 ============
theta_0_0 = 0.476
thought0.0 = flip(theta_0_0)
thought0.1 = not thought0.0
code0.0 = if thought0.0 then 'query = "AI regulation site:arxiv.org submitted June 2022"\nai_regulation_pages = web_search(query=query)\nprint(ai_regulation_pages)' else Error
obs0.0 = execute(code0.0)
factor(-5.018)   # log(0.006615)
code0.1 = if thought0.1 then Error else Error
obs0.1 = execute(code0.1)
factor(-9.210)   # log(0.000000)

{0.0, 0.1} = resample_top_k({0.0, 0.1}, k=2)   # kept: ['0.0', '0.1']

# ============ ROUND 2 ============
theta_0.0_0 = 0.520
thought0.0.0 = flip(theta_0.0_0)
thought0.0.1 = not thought0.0.0
code0.0.0 = if tho